In [1]:
import pandas as pd
import numpy as np
import pulp

In [2]:
df = pd.read_csv('../data/processed/rq3_inputs.csv', index_col='Datetime (UTC)', parse_dates=True)

In [3]:
predicted_prices = df['predicted_price'].values
actual_prices = df['actual_price'].values

predicted_P = df['predicted_P'].values
actual_P = df['actual_P'].values

In [4]:
BATTERY_CAPACITY = 500
MIN_SOC = 0.1 * BATTERY_CAPACITY
MAX_SOC = 0.9 * BATTERY_CAPACITY
MAX_CHARGE_RATE = 100
MAX_DISCHARGE_RATE = 100
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95
HORIZON = 24

In [5]:
def solve_mpc_window(prices, productions, initial_soc, capacity, min_soc, max_soc, max_charge, max_discharge, charge_eff, discharge_eff):

    H = len(prices)
    prob = pulp.LpProblem("battery_mpc", pulp.LpMaximize)

    sell = [pulp.LpVariable(f"sell_{t}", lowBound=0) for t in range(H)]
    charge = [pulp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_charge) for t in range(H)]
    discharge = [pulp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_discharge) for t in range(H)]
    soc = [pulp.LpVariable(f"soc_{t}", lowBound=min_soc, upBound=max_soc) for t in range(H)]

    prob += pulp.lpSum([prices[t] * (sell[t] + discharge[t] * discharge_eff) / 1000 for t in range(H)])

    for t in range(H):
        prob += sell[t] + charge[t] == productions[t]
        if t == 0:
            prob += soc[t] == initial_soc + charge[t] * charge_eff - discharge[t]
        else:
            prob += soc[t] == soc[t-1] + charge[t] * charge_eff - discharge[t]

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return [(sell[t].varValue, charge[t].varValue, discharge[t].varValue) for t in range(H)]

In [6]:
def run_mpc_simulation(predicted_prices, predicted_P, actual_prices, actual_P,
                        capacity, min_soc, max_soc, max_charge, max_discharge,
                        charge_eff, discharge_eff, horizon=24):
    n = len(predicted_prices)
    soc = min_soc
    revenue = 0

    for t in range(n):
        window_end = min(t + horizon, n)
        window_prices = predicted_prices[t:window_end]
        window_prod = predicted_P[t:window_end]

        decisions = solve_mpc_window(
            window_prices, window_prod, soc,
            capacity, min_soc, max_soc,
            max_charge, max_discharge, charge_eff, discharge_eff
        )
        sell_amt, charge_amt, discharge_amt = decisions[0]

        real_price = actual_prices[t]
        real_production = actual_P[t]

        actual_sell = min(sell_amt, real_production)
        actual_charge = min(charge_amt, real_production - actual_sell, max_soc - soc)

        revenue += real_price * (actual_sell + discharge_amt * discharge_eff) / 1000
        soc = soc + actual_charge * charge_eff - discharge_amt

    return revenue

In [7]:
def perturb_random(values, std_pct, seed=None, min_value=0):

    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0, scale=std_pct, size=len(values))
    perturbed = values * (1 + noise)
    return np.clip(perturbed, min_value, None)

In [8]:
base_revenue = run_mpc_simulation(
    predicted_prices, predicted_P, actual_prices, actual_P,
    BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
    MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
    CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY, HORIZON
)
print(f"Base MPC приход: {base_revenue:.2f} EUR")

Base MPC приход: 1111.54 EUR


In [9]:
price_uncertainty_levels = [0.10, 0.20, 0.30]
price_results = {}

for std_pct in price_uncertainty_levels:
    perturbed_prices = perturb_random(predicted_prices, std_pct, seed=42)
    revenue = run_mpc_simulation(
        perturbed_prices, predicted_P, actual_prices, actual_P,
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY, HORIZON
    )
    price_results[std_pct] = revenue
    print(f"Ниво на несигурност во цена +-{std_pct*100:.0f}% -> приход: {revenue:.2f} EUR (разлика од base: {revenue - base_revenue:+.2f})")

Ниво на несигурност во цена +-10% -> приход: 1111.06 EUR (разлика од base: -0.48)
Ниво на несигурност во цена +-20% -> приход: 1062.25 EUR (разлика од base: -49.29)
Ниво на несигурност во цена +-30% -> приход: 1040.74 EUR (разлика од base: -70.80)


In [10]:
production_uncertainty_levels = [0.10, 0.20, 0.30]
production_results = {}

for std_pct in production_uncertainty_levels:
    perturbed_P = perturb_random(predicted_P, std_pct, seed=42)
    revenue = run_mpc_simulation(
        predicted_prices, perturbed_P, actual_prices, actual_P,
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY, HORIZON
    )
    production_results[std_pct] = revenue
    print(f"Ниво на несигурност во производство +-{std_pct*100:.0f}% -> приход: {revenue:.2f} EUR (разлика од base: {revenue - base_revenue:+.2f})")

Ниво на несигурност во производство +-10% -> приход: 1111.57 EUR (разлика од base: +0.03)
Ниво на несигурност во производство +-20% -> приход: 1092.57 EUR (разлика од base: -18.98)
Ниво на несигурност во производство +-30% -> приход: 1055.82 EUR (разлика од base: -55.72)
